# Checkout page redesigns: Dunnett's test

**One-Way Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** An online shop timed how long test users took to complete checkout on the current page and on three redesigns, with five users per version. The designers are sure that Redesign B is faster: in the test, it was about 10 seconds quicker than the current page.

**The question.** Is any redesign really faster than the current page, or should the shop keep what it has?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/one_way_anova__dunnett.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Current page,Redesign A,Redesign B,Redesign C
95.0,102.0,86.0,98.0
106.0,77.0,82.0,106.0
94.0,85.0,91.0,100.0
91.0,96.0,89.0,96.0
91.0,104.0,81.0,83.0

These cells save `eta_squared_interval.py`, `welch_anova.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile eta_squared_interval.py
"""The confidence interval for partial eta squared, from the noncentral F distribution.

SciPy has the noncentral F (stats.ncf) but no function that inverts it for an interval,
so the inversion is written out (Steiger 2004). The interval is found for the
noncentrality λ and converted to partial eta squared as λ / (λ + df1 + df2 + 1).
"""

import numpy as np
from scipy import optimize, stats


def _limit(f, df1, df2, q):
    """The λ ≥ 0 under which the observed F sits at the q quantile of F(df1, df2, λ).

    0 when even λ = 0 cannot put it there: an F too small for the interval to leave 0.
    The cdf at F falls as λ rises, so the bracket opens outwards from F·df1, doubling.
    """
    def gap(lam):
        cdf = stats.f.cdf(f, df1, df2) if lam == 0 else stats.ncf.cdf(f, df1, df2, lam)
        return cdf - q

    if gap(0.0) <= 0:
        return 0.0
    start = max(f * df1, 1.0)
    step = 1.0
    while gap(max(start - step, 0.0)) <= 0:
        step *= 2
    low = max(start - step, 0.0)
    step = 1.0
    while gap(start + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, start + step, xtol=1e-14,
                           rtol=4 * np.finfo(float).eps, maxiter=500)


def partial_eta_squared_interval(f, df1, df2, level):
    """(lower, upper): the level interval for partial eta squared (Steiger 2004)."""
    total = df1 + df2 + 1
    lower = _limit(f, df1, df2, (1 + level) / 2)
    upper = _limit(f, df1, df2, (1 - level) / 2)
    return lower / (lower + total), upper / (upper + total)

In [ ]:
%%writefile welch_anova.py
"""Brown and Forsythe's (1974) one-way ANOVA of means, which does not assume one variance.

Welch's test comes from statsmodels' anova_oneway(use_var="unequal"). Its use_var="bf"
applies Mehrotra's (1997) correction to the first degrees of freedom, and SciPy has no
Brown-Forsythe test of means, so the original test is written out here.
"""

import numpy as np
from scipy import stats


def brown_forsythe(groups):
    """Brown & Forsythe (1974) F*: (F, df1, df2, P), denominator weighted by variance."""
    k = len(groups)
    n = np.array([len(g) for g in groups], dtype=float)
    means = np.array([np.mean(g) for g in groups])
    variances = np.array([np.var(g, ddof=1) for g in groups])
    grand = np.concatenate(groups).mean()
    # The usual between-groups sum of squares, set against each group's variance
    # weighted by 1 - n_i / N instead of one pooled variance.
    numerator = (n * (means - grand) ** 2).sum()
    weights = (1 - n / n.sum()) * variances
    f = numerator / weights.sum()
    # Satterthwaite's degrees of freedom for that weighted denominator.
    c = weights / weights.sum()
    df1, df2 = k - 1, 1 / ((c**2 / (n - 1)).sum())
    return f, float(df1), df2, stats.f.sf(f, df1, df2)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.oneway import anova_oneway
from welch_anova import brown_forsythe

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per group. Blank cells are skipped, so groups may differ in size.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["Current page", "Redesign A", "Redesign B", "Redesign C"]
groups = {name: data[name].dropna().to_numpy(float) for name in levels}
samples = list(groups.values())

## Descriptive statistics

In [ ]:
n = np.array([len(v) for v in samples], dtype=float)
means = np.array([v.mean() for v in samples])
sds = np.array([v.std(ddof=1) for v in samples])
ses = sds / np.sqrt(n)
descriptive = pd.DataFrame(
    {"n": n.astype(int), "mean": means, "SD": sds, "mean - SD": means - sds,
     "mean + SD": means + sds, "SE": ses, "mean - SE": means - ses, "mean + SE": means + ses},
    index=levels,
)
print(descriptive)

## Test for homogeneity of variance

In [ ]:
# The ANOVA assumes every group has the same spread. Both tests ask whether the spreads
# could be equal: a small P value says they probably are not. Bartlett's test assumes
# normal data; Levene's test copes better with non-normal data. center="mean" gives
# Levene's original form; SciPy's default, the median, is the Brown-Forsythe variant.
bartlett = stats.bartlett(*samples)
levene = stats.levene(*samples, center="mean")
print(f"\nBartlett: chi-square = {bartlett.statistic:.4f}, df = {len(samples) - 1}, P = {bartlett.pvalue:.4g}")
print(f"Levene:   F = {levene.statistic:.4f}, df = ({len(samples) - 1}, {int(n.sum()) - len(samples)}), "
      f"P = {levene.pvalue:.4g}")

## Analysis of variance table

In [ ]:
# F = (variation between the group means, per df) / (variation within the groups, per
# df). If every population mean were the same, F would be near 1, and P is the chance
# of an F at least this large in that case. The test assumes independent values,
# normal errors and one common spread. statsmodels fits the one-factor linear model.
k, total_n = len(samples), int(n.sum())
long = pd.DataFrame({"value": np.concatenate(samples),
                     "group": np.repeat(levels, [len(v) for v in samples])})
fit = smf.ols("value ~ C(group)", data=long).fit()
table = anova_lm(fit)
ss_between, ss_within = table["sum_sq"].iloc[0], table["sum_sq"].iloc[1]
df_between, df_within = int(table["df"].iloc[0]), int(table["df"].iloc[1])
ms_between, ms_within = table["mean_sq"].iloc[0], table["mean_sq"].iloc[1]
f_value, p_value = table["F"].iloc[0], table["PR(>F)"].iloc[0]
anova = pd.DataFrame(
    [[ss_between, df_between, ms_between, f_value, p_value],
     [ss_within, df_within, ms_within, np.nan, np.nan],
     [ss_between + ss_within, total_n - 1, np.nan, np.nan, np.nan]],
    index=["Between groups", "Error", "Total"], columns=["SS", "df", "MS", "F", "P"],
)
print()
print(anova)

## Tests that do not assume equal variances

In [ ]:
# Read these instead of the table above when the variances differ. statsmodels'
# anova_oneway(use_var="unequal") is Welch's test. Its use_var="bf" applies Mehrotra's
# (1997) correction to the first df, so Brown and Forsythe's original test is written
# out in welch_anova.py.
welch = anova_oneway(long["value"], long["group"], use_var="unequal")
unequal = pd.DataFrame(
    [[welch.statistic, welch.df[0], welch.df[1], welch.pvalue], brown_forsythe(samples)],
    index=["Welch", "Brown-Forsythe"], columns=["F", "df1", "df2", "P"],
)
print()
print(unequal)

## Multiple comparison: Dunnett

In [ ]:
# Every group against the control only, two-sided and in each direction. Fewer
# comparisons than every pair, so each is tested with more power. SciPy integrates the
# multivariate t by quasi-Monte Carlo, so its P values agree with the app's to about
# 1% rather than exactly; the app integrates the same distribution deterministically.
# The control is the level the settings name, spaces at either end aside, as the app
# reads it; the first level when they name none.
named = ("Current page" or "").strip()
control = next((name for name in levels if name.strip() == named), levels[0])
others = [name for name in levels if name != control]
rows = []
for label, alternative, sign in (("control ≠", "two-sided", 1), ("control <", "greater", -1),
                                 ("control >", "less", -1)):
    result = stats.dunnett(*[groups[name] for name in others], control=groups[control],
                           alternative=alternative, random_state=1)
    for name, p in zip(others, result.pvalue):
        difference = groups[name].mean() - groups[control].mean()
        se = np.sqrt(ms_within * (1 / len(groups[control]) + 1 / len(groups[name])))
        rows.append([label, control, name, groups[control].mean(), groups[name].mean(),
                     sign * difference, se, sign * difference / se, p])
pairs = pd.DataFrame(rows, columns=["alternative", "level 1", "level 2", "mean 1", "mean 2",
                                    "difference", "SE", "statistic", "P"])
print()
print(pairs.to_string(index=False))

## Effect sizes

In [ ]:
# eta² = SS between / SS total, and partial eta² = SS between / (SS between + SS error),
# which is the same number when there is one factor. omega² = (SS between - df·MS error)
# / (SS total + MS error) removes eta²'s upward bias (Hays 1963). Partial eta² carries the
# 95% interval that inverts the noncentral F at the observed F (Steiger 2004), written out
# in eta_squared_interval.py.
from eta_squared_interval import partial_eta_squared_interval


def size(value):
    """Cohen's (1988) rule of thumb for an eta²: 0.01 small, 0.06 medium, 0.14 large."""
    return ["negligible", "small", "medium", "large"][sum(value >= s for s in (0.01, 0.06, 0.14))]


ss_total = ss_between + ss_within
lower, upper = partial_eta_squared_interval(f_value, df_between, df_within, 0.95)
effect_sizes = pd.DataFrame(
    [[ss_between / ss_total, np.nan, np.nan],
     [ss_between / (ss_between + ss_within), lower, upper],
     [(ss_between - df_between * ms_within) / (ss_total + ms_within), np.nan, np.nan]],
    index=["eta²", "partial eta²", "omega²"], columns=["value", "lower", "upper"],
)
effect_sizes["size"] = effect_sizes["value"].map(size)
print()
print(effect_sizes)

## Normality of the residuals: the Shapiro-Wilk test

In [ ]:
# The residuals are each value minus its group's mean -- the errors the ANOVA assumes
# normal -- rather than the values themselves, which mix the groups' different means.
# A small P value is evidence against normality; a large one is not proof of it.
residuals = fit.resid.to_numpy()
shapiro = stats.shapiro(residuals)
normality = [len(residuals), shapiro.statistic, shapiro.pvalue]
print(f"\nShapiro-Wilk on the residuals: n = {len(residuals)}, W = {shapiro.statistic:.4f}, "
      f"P = {shapiro.pvalue:.4g}")

## What the result says

Redesign B looks faster: its mean is 85.8 seconds against 95.4 for the current page. The analysis of variance asks whether the four pages differ at all. F compares the gaps between the page means with the spread between users: F = 1.79 and P = 0.190. The P value is the chance of gaps at least this large if the pages were truly equal.

Dunnett's test compares each redesign with one control, the current page, and corrects for making three comparisons. For Redesign B the difference is 9.6 seconds and the two-sided P value is 0.185.

With five users per page, a gap of 9.6 seconds could easily be luck. A real difference is not shown, so keep the current page for now.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.